<a href="https://colab.research.google.com/github/FarsusDasdana/similarity_search/blob/main/Embedding_SimSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers

In [ ]:
from collections import OrderedDict
from datetime import datetime
from sentence_transformers import util
import torch
import os


def get_most_similar_name(
    name_list,
    input_name,
    model,
    embedding_path: str = "model.pt"
):
    """
    Finds the most similar name in `name_list` to `input_name` using sentence-transformers.

    Args:
        name_list (list of str): List of names to compare.
        input_name (str): Name to match.
        model (SentenceTransformer): Hugging Face model to use for embeddings.
        embedding_path (str): Path to save the embedding of the input name.

    Returns:
        OrderedDict: {name: similarity_score} sorted from highest to lowest score.
    """

    # Step 1: Load or compute input embedding
    start_time = datetime.now()
    input_embedding = model.encode(input_name, convert_to_tensor=True)
    end_time = datetime.now()
    print(f"[1] Input embedding processing duration: {end_time - start_time}")

    # Step 2: Encode name list
    start_time = datetime.now()
    if os.path.exists(embedding_path):
      name_embeddings = torch.load(embedding_path)
      print("Embeddings loaded")
    else:
      name_embeddings = model.encode(name_list, convert_to_tensor=True)
      torch.save(name_embeddings, embedding_path)
      print("Embeddings saved")
    end_time = datetime.now()
    print(f"[2] Name list encoding duration: {end_time - start_time}")

    # Step 3: Compute cosine similarity
    start_time = datetime.now()
    cosine_scores = util.pytorch_cos_sim(input_embedding, name_embeddings)[0]
    end_time = datetime.now()
    print(f"[3] Cosine similarity computation duration: {end_time - start_time}")

    # Step 4: Sort results
    start_time = datetime.now()
    name_score_pairs = {name: round(score.item(), 4) for name, score in zip(name_list, cosine_scores)}
    sorted_scores = OrderedDict(
        sorted(name_score_pairs.items(), key=lambda x: x[1], reverse=True)
    )
    end_time = datetime.now()
    print(f"[4] Sorting results duration: {end_time - start_time}")

    return sorted_scores


In [ ]:
from sentence_transformers import SentenceTransformer

# Requires personal token: https://huggingface.co/docs/hub/security-tokens
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
import pandas as pd
import random
from datetime import datetime


# Read first 5000 rows for simplicity
df = pd.read_csv("/content/100000_rows.csv", sep="\t", nrows=5000)  # Path will be change from user to user
# Concat names and addresses to generate single list
name_list = list(df["name"] + ":  " + df["street_address"])

# Select random name in name list to test the function
input_name = random.choice(name_list)

# Calculate the similarity scores
start_time = datetime.now()
sim_dict = get_most_similar_name(
    name_list = name_list,
    input_name = input_name,
    model = embedding_model
)
end_time = datetime.now()

print(f"Input Name: {input_name}")
print(f"Top - 5 similar names: {dict(list(sim_dict.items())[:5])}")
print('Total Duration: {}'.format(end_time - start_time))

[1] Input embedding processing duration: 0:00:00.068004
Embeddings saved
[2] Name list encoding duration: 0:01:05.113894
[3] Cosine similarity computation duration: 0:00:00.006813
[4] Sorting results duration: 0:00:00.044335
Input Name: Feriendorf Am Vogelpark Marlow:  18337 , Marlow
Top - 5 similar names: {'Feriendorf Am Vogelpark Marlow:  18337 , Marlow': 1.0, 'Ferienhaus Margreth:  Schneilstraße 6': 0.6121, 'Gästehaus Ferienpark Sonntag:  Winterberger Str. 17 , 59964 , Medebach': 0.5369, 'Hotel Am Vogelsanger Weg:  Vogelsanger Weg 36': 0.5307, 'Hotel Margarit:  C/Ultonia, 1': 0.5297}
Total Duration: 0:01:05.235685


In [ ]:
# SECOND CALL! (AFTER CREATING EMBEDDING VALUES FINDING BEST MATCH IS EASY!)

# Select random name in name list to test the function
input_name = random.choice(name_list)

# Calculate the similarity scores
start_time = datetime.now()
sim_dict = get_most_similar_name(
    name_list = name_list,
    input_name = input_name,
    model = embedding_model
)
end_time = datetime.now()

print(f"Input Name: {input_name}")
print(f"Top - 5 similar names: {dict(list(sim_dict.items())[:5])}")
print('Total Duration: {}'.format(end_time - start_time))

[1] Input embedding processing duration: 0:00:00.033745
Embeddings loaded
[2] Name list encoding duration: 0:00:00.005881
[3] Cosine similarity computation duration: 0:00:00.007284
[4] Sorting results duration: 0:00:00.030141
Input Name: Alpha Travel Consultants Gmbh:  Berlin Germany
Top - 5 similar names: {'Alpha Travel Consultants Gmbh:  Berlin Germany': 1.0, 'Avenon Privat-hotel Am Steinberg:  Werner-von-Siemens-Allee, Roethenbach an der Pegnitz, BY, 90552, Germany': 0.4996, 'FREIgeist Göttingen Innenstadt, A Member of Design Hotels:  BERLINER STR. 30': 0.494, 'Nordic Hotels Luebeck Gmbh:  Ahrensboker Strasse 4, 23554 Lubeck, Germany': 0.4804, 'Arcadia Hotel Berlin:  Frankfurter Allee 73a': 0.4702}
Total Duration: 0:00:00.078239
